### Import Packages

In [1]:
import duckdb
from pathlib import Path

#### Configs and Variables

In [2]:
data_vault_path = r"C:\Users\JacksonDivakarRajasi\OneDrive - JMAN Group Ltd\Documents\Python Folder\DuckDB Environment\Olist Eccomerce DuckDB\DuckDB Data\data_vault.duckdb"

#### DuckDB Connection

In [3]:
con = duckdb.connect(data_vault_path)

In [4]:
con.execute("""
CREATE SCHEMA IF NOT EXISTS business_vault;
""")

## Data Mapping

In [5]:
con.execute("""
CREATE OR REPLACE TABLE business_vault.product_enriched AS
SELECT
    s.product_hk,
    s.product_category_name,
    t.product_category_name_english,

    s.product_name_length,
    s.product_description_length,
    s.product_photos_qty,
    s.product_weight_g,
    s.product_length_cm,
    s.product_height_cm,
    s.product_width_cm,

    s.load_date,
    s.record_source

FROM raw_vault.sat_product_details s

LEFT JOIN prep.product_category_translation t
    ON s.product_category_name = t.product_category_name;
""")

## Geolocation Enrichment

### Customer Enrichment

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE business_vault.customer_enriched_history AS
SELECT
    c.customer_hk,
    c.customer_unique_id,
    c.customer_zip_code_prefix,
    c.customer_city,
    c.customer_state,

    g.geolocation_lat AS latitude,
    g.geolocation_lng AS longitude,

    c.hash_diff,
    c.load_date,
    c.record_source

FROM raw_vault.sat_customer_details c

LEFT JOIN prep.geolocation g
    ON c.customer_zip_code_prefix =
       g.geolocation_zip_code_prefix;
""")

In [7]:
con.execute("""
CREATE OR REPLACE VIEW business_vault.customer_enriched_current AS
SELECT *
FROM business_vault.customer_enriched_history
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY customer_hk
    ORDER BY load_date DESC
) = 1;
""")

### Seller Enrichment

In [8]:
con.execute("""
CREATE OR REPLACE TABLE business_vault.seller_enriched_history AS
SELECT
    s.seller_hk,
    s.seller_zip_code_prefix,
    s.seller_city,
    s.seller_state,

    g.geolocation_lat AS latitude,
    g.geolocation_lng AS longitude,

    s.hash_diff,
    s.load_date,
    s.record_source

FROM raw_vault.sat_seller_details s

LEFT JOIN prep.geolocation g
    ON s.seller_zip_code_prefix =
       g.geolocation_zip_code_prefix;
""")

In [9]:
con.execute("""
CREATE OR REPLACE VIEW business_vault.seller_enriched_current AS
SELECT *
FROM business_vault.seller_enriched_history
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY seller_hk
    ORDER BY load_date DESC
) = 1;
""")

## Product Enriched

In [10]:
con.execute("""
CREATE OR REPLACE TABLE business_vault.product_enriched_history AS
SELECT
    p.product_hk,

    p.product_category_name,
    t.product_category_name_english,

    p.product_name_length,
    p.product_description_length,
    p.product_photos_qty,
    p.product_weight_g,
    p.product_length_cm,
    p.product_height_cm,
    p.product_width_cm,

    p.hash_diff,
    p.load_date,
    p.record_source

FROM raw_vault.sat_product_details p

LEFT JOIN prep.product_category_translation t
    ON p.product_category_name =
       t.product_category_name;
""")

In [11]:
con.execute("""
CREATE OR REPLACE VIEW business_vault.product_enriched_current AS
SELECT *
FROM business_vault.product_enriched_history
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY product_hk
    ORDER BY load_date DESC
) = 1;
""")

## Order Lifecycle Business Vault Table

In [12]:
con.execute("""
CREATE OR REPLACE TABLE business_vault.order_lifecycle_history AS
SELECT
    order_hk,
    order_status,

    order_purchase_timestamp,
    order_approved_at,
    order_delivered_carrier_date,
    order_delivered_customer_date,
    order_estimated_delivery_date,

    DATE_DIFF(
        'hour',
        order_purchase_timestamp,
        order_approved_at
    ) AS hours_to_approve,

    DATE_DIFF(
        'hour',
        order_purchase_timestamp,
        order_delivered_carrier_date
    ) AS hours_to_carrier,

    DATE_DIFF(
        'day',
        order_purchase_timestamp,
        order_delivered_customer_date
    ) AS days_to_deliver,

    DATE_DIFF(
        'day',
        order_estimated_delivery_date,
        order_delivered_customer_date
    ) AS delivery_delay_days,

    CASE
        WHEN order_delivered_customer_date IS NULL
            THEN 'not_delivered'
        WHEN order_delivered_customer_date
             <= order_estimated_delivery_date
            THEN 'on_time'
        ELSE 'late'
    END AS delivery_status,

    hash_diff,
    load_date,
    record_source

FROM raw_vault.sat_order_details;
""")

In [13]:
con.execute("""
CREATE OR REPLACE VIEW business_vault.order_lifecycle_current AS
SELECT *
FROM business_vault.order_lifecycle_history
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY order_hk
    ORDER BY load_date DESC
) = 1;
""")

## Current Tables

In [14]:
# =========================================================
# CURRENT ORDER
# =========================================================
con.execute("""
CREATE OR REPLACE VIEW business_vault.current_order AS
SELECT *
FROM raw_vault.sat_order_details
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY order_hk
    ORDER BY load_date DESC
) = 1;
""")


# =========================================================
# CURRENT REVIEW
# =========================================================
con.execute("""
CREATE OR REPLACE VIEW business_vault.current_review AS
SELECT *
FROM raw_vault.sat_review_details
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY review_hk
    ORDER BY load_date DESC
) = 1;
""")


# =========================================================
# CURRENT ORDER ITEM
# =========================================================
con.execute("""
CREATE OR REPLACE VIEW business_vault.current_order_item AS
SELECT *
FROM raw_vault.sat_order_item_details
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY order_item_hk
    ORDER BY load_date DESC
) = 1;
""")


# =========================================================
# CURRENT ORDER PAYMENT
# =========================================================
con.execute("""
CREATE OR REPLACE VIEW business_vault.current_order_payment AS
SELECT *
FROM raw_vault.sat_order_payment_details
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY order_payment_hk
    ORDER BY load_date DESC
) = 1;
""")